# Team 18 (`TEAM_ID = 23`) — Kaggle / Google Colab Data Engine Runner

Notebook chạy 1-click toàn bộ quy trình **Thu thập ảnh VN (Script #1) → Pre-label SAM2 xuất CVAT (Script #2) → Chuẩn hóa Public IDD/Mapillary (Script #3) → Auto-QC đóng gói `dataset_vX.zip` (Script #4)**.

**Khuyến nghị GPU (Best Choice):**
- **Kaggle Notebooks (Free):** `Settings > Accelerator > GPU P100 hoặc GPU T4 x2`, bật `Internet = ON`, gắn thêm dataset `manjotpahwa/indian-driving-dataset` ở mục `+ Add Input` (0 giây tải!).
- **Google Colab:** `Runtime > Change runtime type > T4 GPU` (Free) hoặc `L4 GPU` (Pro).

In [ ]:
#@title 1. Cài đặt thư viện Data Engine (chạy 1 lần đầu phiên)
!pip install -q shapely pyyaml opencv-python-headless yt-dlp datasets huggingface-hub ultralytics
import torch, os
print('CUDA available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
#@title 2. Khai báo thư mục project `robotaxi-hackathon-team18`
# Nếu chạy trên Colab: upload thư mục `robotaxi-hackathon-team18` lên Google Drive rồi mount.
# Nếu chạy trên Kaggle: upload thư mục `scripts/` + `configs/` thành Private Kaggle Dataset rồi copy sang `/kaggle/working/proj`.
from pathlib import Path
IN_COLAB = 'COLAB_GPU' in os.environ or Path('/content').exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJ = Path('/content/drive/MyDrive/robotaxi-hackathon-team18')
else:
    PROJ = Path('/kaggle/working/robotaxi-hackathon-team18')

WORK = Path('/content/work_data' if IN_COLAB else '/kaggle/working/work_data')
WORK.mkdir(parents=True, exist_ok=True)
print('PROJ =', PROJ, '| WORK =', WORK)

In [ ]:
#@title 3. [Script #1] Gom ảnh đường phố Việt Nam ngay tại phòng (YouTube 4K Dashcam + HF VN Traffic Sign)
# (a) Trích 300 ảnh VN thật + GT bbox biển báo VN từ HuggingFace:
!python {PROJ}/scripts/01_vn_frame_harvester.py \
    --hf-vn-signs \
    --max-hf-images 300 \
    --out {WORK}/vn_frames

# (b) Sau khi điền link video YouTube 4K VN vào configs/vn_youtube_sources.yaml, bỏ comment dòng dưới để chạy:
# !python {PROJ}/scripts/01_vn_frame_harvester.py --config {PROJ}/configs/vn_youtube_sources.yaml --out {WORK}/vn_frames

In [ ]:
#@title 4. [Script #2] Chạy Auto Pre-label (YOLO11x-seg + YOLO-Worldv2 + SAM2) -> Xuất file Zip nạp vào CVAT
# Chạy cho lô ảnh cần gán (ví dụ 200 ảnh đầu tiên hoặc toàn bộ tập val):
!python {PROJ}/scripts/02_auto_prelabel_sam2.py \
    --images {WORK}/vn_frames/train/images \
    --extra-sign-boxes {WORK}/vn_frames/train/extra_sign_boxes \
    --out {WORK}/cvat_vn_batch1_prelabel \
    --max-images 200 \
    --use-yoloworld \
    --use-sam2

print('Tải tệp cvat_yolo_seg_import.zip lên CVAT để cả đội sửa nhanh!')

In [ ]:
#@title 5. [Script #3] Chuyển đổi Public Dataset (IDD trên Kaggle & Mapillary Vistas v2 HF) sang 11 lớp + Rider-Bike Union
# (a) IDD trên Kaggle (0 giây tải khi đã Add Input `manjotpahwa/indian-driving-dataset`):
# !python {PROJ}/scripts/03_public_converter.py \
#     --source idd \
#     --input /kaggle/input/indian-driving-dataset \
#     --out {WORK}/public_idd \
#     --max-images 600

# (b) Mapillary Vistas v2 streaming trực tiếp từ HuggingFace:
!python {PROJ}/scripts/03_public_converter.py \
    --source mapillary_hf \
    --out {WORK}/public_mapillary \
    --max-images 1000 \
    --max-scan 4000 \
    --min-priority 10.0

In [ ]:
#@title 6. [Script #4] Auto-QC Gate + Khử Overlap + Chống Leakage + Đóng gói `dataset_v1.zip`
# Chỉnh đường dẫn các nguồn sạch (Public + VN đã QC trên CVAT + Golden VN Val) rồi chạy:
!python {PROJ}/scripts/04_auto_qc_and_packager.py \
    --train-dirs {WORK}/public_mapillary {WORK}/public_idd {WORK}/cvat_vn_train_clean \
    --val-dir {WORK}/cvat_golden_val_clean \
    --out-zip {WORK}/dataset_v1.zip